# Step 1 — Neo4j Setup & Graph Schema

**SignalPulse AI** stores everything in a single Neo4j database that acts as *both* a knowledge graph **and** a vector store. In this notebook we will:

1. Connect to the running Neo4j server.
2. Learn the core graph-database concepts (nodes, relationships, Cypher).
3. Define our **schema**: the `Document`, `Chunk`, and `Entity` node types.
4. Create **uniqueness constraints**, **vector indexes**, and **full-text indexes**.
5. Verify everything — both in Python and in the Neo4j Browser.

> **Prerequisite:** Neo4j must be running. If it isn't, run `.\start_neo4j.ps1` from the project root first.
>
> All the reusable database code lives in `signalpulse/graph.py`. This notebook *explains* the concepts and *calls* that code — so nothing here is throwaway.

## 1. Graph database concepts (the basics)

A **graph database** stores data as a *network* instead of tables of rows and columns.

- **Node** — a "thing" (imagine a circle). Examples: a document, a text chunk, an agency.
- **Label** — the *type* of a node, written with a colon: `:Document`, `:Chunk`, `:Entity`. Labels let us find nodes of a kind quickly.
- **Property** — a key/value stored on a node or relationship. Examples: `title`, `url`, `embedding`.
- **Relationship (edge)** — a *directed, typed* connection between two nodes, drawn as an arrow: `(:Document)-[:HAS_CHUNK]->(:Chunk)`.
- **Cypher** — Neo4j's query language. It reads like ASCII-art of the pattern you want: `(a)-[:REL]->(b)`. Key verbs: `MATCH` (find a pattern), `MERGE` (create only if missing), `CREATE` (always create), `SHOW` (inspect schema).

### Our schema

```
 (:Document) --HAS_CHUNK--> (:Chunk) --MENTIONS--> (:Entity)
      |                        |                       |
 summary+embedding        text+embedding        (:Entity)-[:RELATED_TO {type}]->(:Entity)
```

| Node | Unique key | Other properties |
|---|---|---|
| `Document` | `id` | `title`, `url`, `agency`, `domain`, `published_date`, `content_hash`, `summary`, `embedding` |
| `Chunk` | `id` | `text`, `chunk_index`, `embedding` |
| `Entity` | `name` | `type` |

**Relationships:** `HAS_CHUNK` (Document→Chunk), `MENTIONS` (Chunk→Entity), `RELATED_TO` (Entity→Entity, carries a `type` such as *AFFECTS* or *REQUIRES*).

In [1]:
import sys
from pathlib import Path

# Make the project root importable, whether the kernel starts in notebooks/ or the root.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from signalpulse import graph
from signalpulse.config import settings

print("Project root :", ROOT)
print("Neo4j URI    :", settings.NEO4J_URI)
print("Embedding dim:", settings.EMBEDDING_DIM)

Project root : C:\Users\saihaj\Documents\22nd Project
Neo4j URI    : bolt://localhost:7687
Embedding dim: 384


## 2. Connect to Neo4j

`signalpulse.graph` keeps a single shared **driver** — a thread-safe connection pool — that all our code reuses. `verify_connectivity()` throws if the database is unreachable, so it's a good first check. Then we ask the server which version/edition it is.

In [2]:
graph.verify_connectivity()

info = graph.run_query(
    "CALL dbms.components() YIELD name, versions, edition "
    "RETURN name, versions[0] AS version, edition"
)
print("Connected!")
for row in info:
    print(f"  {row['name']} {row['version']} ({row['edition']} edition)")

Connected!
  Neo4j Kernel 5.26.0 (community edition)


## 3. Uniqueness constraints

A **constraint** is a rule the database enforces. A **uniqueness constraint** guarantees that no two nodes of a label can share the same key value — e.g. two `Document` nodes can never have the same `id`.

Why we need them:
- **Data integrity** — the scheduled pipeline runs repeatedly; constraints stop it from creating duplicate documents/chunks/entities.
- **Speed** — each uniqueness constraint is backed by an index, so `MERGE (d:Document {id: ...})` (our "create-or-update" operation) is fast.

We create three: `Document.id`, `Chunk.id`, and `Entity.name` must each be unique. Every statement uses `IF NOT EXISTS`, so re-running is harmless (**idempotent**).

In [3]:
created = graph.create_constraints()
print("Constraints ensured:", created)
print()

for c in graph.show_constraints():
    print(f"  - {c['name']:<14} on {c.get('labelsOrTypes')} {c.get('properties')}  ({c['type']})")

Constraints ensured: ['document_id', 'chunk_id', 'entity_name']

  - chunk_id       on ['Chunk'] ['id']  (UNIQUENESS)
  - document_id    on ['Document'] ['id']  (UNIQUENESS)
  - entity_name    on ['Entity'] ['name']  (UNIQUENESS)


## 4. Embeddings & the vector index

An **embedding** is a list of numbers (a *vector*) that captures the *meaning* of a piece of text. Similar meanings → nearby vectors. Our local model `bge-small-en-v1.5` produces **384-dimensional** vectors (that's the `EMBEDDING_DIM` in config).

A **vector index** lets Neo4j find, very quickly, the stored vectors that are *closest* to a query vector — this is **semantic search**. We measure closeness with **cosine similarity** (a score from -1 to 1; higher = more similar meaning).

We create two vector indexes:
- `chunk_embedding` — on `Chunk.embedding` (the main one; we search chunks).
- `document_embedding` — on `Document.embedding` (coarse, document-level matching).

The index must know the exact **dimension** (384) and **similarity function** (`cosine`) up front, which is why `create_vector_indexes()` reads `settings.EMBEDDING_DIM`.

> Note: a vector index only *contains* a node once that node actually has an `embedding` property. Right now there's no data, so the indexes exist but are empty — that's expected. We'll populate them in Step 3.

In [4]:
print(f"Creating vector indexes sized to {settings.EMBEDDING_DIM} dimensions (cosine)...")
created = graph.create_vector_indexes()
print("Vector indexes ensured:", created)
print()

for i in graph.show_indexes():
    if i["type"] == "VECTOR":
        print(f"  - {i['name']:<20} on {i.get('labelsOrTypes')} {i.get('properties')}  state={i.get('state')}")

Creating vector indexes sized to 384 dimensions (cosine)...
Vector indexes ensured: ['chunk_embedding', 'document_embedding']

  - chunk_embedding      on ['Chunk'] ['embedding']  state=ONLINE
  - document_embedding   on ['Document'] ['embedding']  state=ONLINE


## 5. Full-text index

Vector search is great for *meaning*, but sometimes we need an **exact keyword** match — e.g. a specific `CVE-2024-1234`, a memo number, or a regulation citation. Embeddings are weak at those; a **full-text index** (built on Apache Lucene, inside Neo4j) is perfect for them.

We create two:
- `chunk_fulltext` — indexes `Chunk.text`.
- `document_fulltext` — indexes `Document.title` and `Document.summary`.

Together with the vector index, this gives our agent both **semantic** and **exact** retrieval — the "hybrid" search we described in the project outline.

In [5]:
created = graph.create_fulltext_indexes()
print("Full-text indexes ensured:", created)
print()

for i in graph.show_indexes():
    if i["type"] == "FULLTEXT":
        print(f"  - {i['name']:<20} on {i.get('labelsOrTypes')} {i.get('properties')}  state={i.get('state')}")

Full-text indexes ensured: ['chunk_fulltext', 'document_fulltext']

  - chunk_fulltext       on ['Chunk'] ['text']  state=ONLINE
  - document_fulltext    on ['Document'] ['title', 'summary']  state=ONLINE


## 6. Full schema report

In practice you can create the whole schema in one call — `graph.create_schema()` runs the constraints, vector indexes, and full-text indexes together (all idempotent). Below we print a complete summary of what now exists in the database.

In [6]:
# One call creates everything (safe to re-run).
graph.create_schema()

print("=== CONSTRAINTS ===")
for c in graph.show_constraints():
    print(f"  {c['name']:<16} {c['type']:<22} {c.get('labelsOrTypes')} {c.get('properties')}")

print("\n=== INDEXES ===")
for i in graph.show_indexes():
    print(f"  {i['name']:<20} {i['type']:<10} state={i.get('state')}  {i.get('labelsOrTypes')} {i.get('properties')}")

print("\nNode counts:", graph.node_counts())

=== CONSTRAINTS ===
  chunk_id         UNIQUENESS             ['Chunk'] ['id']
  document_id      UNIQUENESS             ['Document'] ['id']
  entity_name      UNIQUENESS             ['Entity'] ['name']

=== INDEXES ===
  chunk_embedding      VECTOR     state=ONLINE  ['Chunk'] ['embedding']
  chunk_fulltext       FULLTEXT   state=ONLINE  ['Chunk'] ['text']
  chunk_id             RANGE      state=ONLINE  ['Chunk'] ['id']
  document_embedding   VECTOR     state=ONLINE  ['Document'] ['embedding']
  document_fulltext    FULLTEXT   state=ONLINE  ['Document'] ['title', 'summary']
  document_id          RANGE      state=ONLINE  ['Document'] ['id']
  entity_name          RANGE      state=ONLINE  ['Entity'] ['name']
  index_343aff4e       LOOKUP     state=ONLINE  None None
  index_f7700477       LOOKUP     state=ONLINE  None None

Node counts: {'Document': 0, 'Chunk': 0, 'Entity': 0}


## 7. (Optional) Create a tiny demo subgraph to visualize

The database is empty, so there's nothing to *see* yet. Let's create a small example — one `Document`, one `Chunk`, and two `Entity` nodes wired together exactly like our schema — so we can visualize it in the Neo4j Browser. This uses `MERGE` (create-or-match), which relies on the uniqueness constraints we just made.

We'll remove this demo data in the last cell so the database is clean for the real pipeline.

In [7]:
graph.run_query(
    """
    MERGE (d:Document {id: 'demo-doc-1'})
      SET d.title = 'DEMO: CISA Alert AA26-001',
          d.agency = 'CISA',
          d.domain = 'Cybersecurity',
          d.summary = 'Demonstration document used to verify the schema.'
    MERGE (c:Chunk {id: 'demo-doc-1::0'})
      SET c.text = 'A critical vulnerability affects the Example Web Server.',
          c.chunk_index = 0
    MERGE (e1:Entity {name: 'Example Web Server'}) SET e1.type = 'Technology'
    MERGE (e2:Entity {name: 'CISA'})               SET e2.type = 'Agency'
    MERGE (d)-[:HAS_CHUNK]->(c)
    MERGE (c)-[:MENTIONS]->(e1)
    MERGE (c)-[:MENTIONS]->(e2)
    MERGE (e2)-[:RELATED_TO {type: 'ISSUED_ALERT_ABOUT'}]->(e1)
    """
)
print("Demo subgraph created. Node counts:", graph.node_counts())
print()

# Read it back to confirm the relationships wired up correctly.
rows = graph.run_query(
    """
    MATCH (d:Document {id: 'demo-doc-1'})-[:HAS_CHUNK]->(c)-[:MENTIONS]->(e)
    RETURN d.title AS document, c.text AS chunk, collect(e.name) AS entities
    """
)
for r in rows:
    print(r)

Demo subgraph created. Node counts: {'Document': 1, 'Chunk': 1, 'Entity': 2}



{'document': 'DEMO: CISA Alert AA26-001', 'chunk': 'A critical vulnerability affects the Example Web Server.', 'entities': ['CISA', 'Example Web Server']}


## 8. Verify in the Neo4j Browser

The Neo4j Browser is a visual tool that ships with the server. To confirm everything by eye:

1. Open **http://localhost:7474** in your web browser.
2. Log in with:
   - **Connect URL:** `bolt://localhost:7687`
   - **Username:** `neo4j`
   - **Password:** `signalpulse123`
3. In the query bar at the top, run these one at a time:

**See the constraints:**
```cypher
SHOW CONSTRAINTS
```

**See the indexes (you should see 2 VECTOR + 2 FULLTEXT + the constraint-backed ones):**
```cypher
SHOW INDEXES
```

**Visualize the demo subgraph** (if you ran the optional cell above):
```cypher
MATCH (n) RETURN n
```
You should see a `Document` connected to a `Chunk`, which points to two `Entity` nodes (`CISA` and `Example Web Server`), with a `RELATED_TO` edge between the entities. Click any node to inspect its properties.

**Test the full-text index directly:**
```cypher
CALL db.index.fulltext.queryNodes('chunk_fulltext', 'vulnerability') YIELD node, score
RETURN node.text AS text, score
```

## 9. Clean up the demo data

Remove the demo nodes so the database is empty and ready for the real ingestion pipeline (Step 2+). The **schema stays** — only the demo *data* is deleted.

> Skip / comment out this cell if you'd like to keep the demo nodes to explore in the browser first.

In [8]:
graph.run_query(
    """
    MATCH (n)
    WHERE n.id IN ['demo-doc-1', 'demo-doc-1::0']
       OR n.name IN ['Example Web Server', 'CISA']
    DETACH DELETE n
    """
)
print("Demo data removed. Node counts:", graph.node_counts())

# Close the shared driver now that the notebook is done.
graph.close_driver()
print("Driver closed. Step 1 complete.")

Demo data removed. Node counts: {'Document': 0, 'Chunk': 0, 'Entity': 0}
Driver closed. Step 1 complete.


## Recap & what's next

In this step we:
- Connected to the local Neo4j server via `signalpulse.graph`.
- Learned nodes, labels, properties, relationships, and Cypher.
- Defined the `Document` / `Chunk` / `Entity` schema.
- Created **3 uniqueness constraints**, **2 vector indexes** (384-dim, cosine), and **2 full-text indexes** — all idempotent and reusable from `graph.create_schema()`.
- Verified everything in Python and in the Neo4j Browser.

**Next — Step 2:** build the data **connectors** that fetch real documents from CISA, NVD, the Federal Register, and others, and normalize them into a common `RawDocument` object ready to load into this schema.